In [ ]:
 # if not yet installed remove # and execute the command

#  pip install pyotp
#  pip install qrcode
#  pip install bcrypt

In [ ]:
import qrcode
from IPython.display import Image, display

# Function to generate and show QR code in Jupyter
def generate_qr(data: str, box_size: int = 10, border: int = 4,
                fill_color: str = "black", back_color: str = "white",
                filename: str = None):
    """
    Generate a QR code and display it in Jupyter Notebook.

    Parameters:
        data (str): Text or URL to encode.
        box_size (int): Size of each box in pixels.
        border (int): Border thickness (in boxes).
        fill_color (str): QR code color.
        back_color (str): Background color.
        filename (str, optional): If provided, saves the QR to this file.
    """
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=box_size,
        border=border,
    )
    qr.add_data(data)
    qr.make(fit=True)

    img = qr.make_image(fill_color=fill_color, back_color=back_color)

    if filename:
        img.save(filename)
        print(f"QR code saved as {filename}")

    # Show inside Jupyter
    display(img)

# Creating a QRcode for the following:

# 1. URL = https//facebook.com
# 2. Telephone = tel:09123456789
# 3. SMS = smsto:09123456789:Hellooo, Musta na u?
# 4. Enail = mailto:student@example.com:Testing Email
# 5. Text = Hello, I Love IT312 - Intregrative Programming
# 6. Geolocation = geo:0.0000,0.0000?site=DNSC



qrData = "https://yahoo.com"


# Example usage
generate_qr(qrData, box_size=8, border=2)

In [ ]:
import pyotp
import qrcode
from IPython.display import display

# Step 1: Generate a secret key (user-specific; should be saved securely)
secret = pyotp.random_base32()
print("Your secret key (save this securely):", secret)

# Step 2: Generate a provisioning URI (to add to Google Authenticator app)
issuer_name = "Sample OTP Generator"      # Name of your app
account_name = "IT312@example.com"  # User identifier (e.g., email)
uri = pyotp.totp.TOTP(secret).provisioning_uri(
    name=account_name,
    issuer_name=issuer_name
)
print("Provisioning URI:", uri)

# Step 3: Generate QR Code for scanning with Google Authenticator
img = qrcode.make(uri)
display(img)


In [ ]:
import pyotp

# Create TOTP object with the saved secret
totp = pyotp.TOTP(secret)

# Simulate user entering OTP from Google Authenticator app
user_otp = input("Enter the 6-digit code from Google Authenticator: ")

if totp.verify(user_otp):
    print("✅ Login successful!")
else:
    print("❌ Invalid code. Try again.")

In [ ]:
import bcrypt
import pyotp
import qrcode
from IPython.display import display

# -------------------------
# USER REGISTRATION (one-time setup)
# -------------------------

# Store user credentials and secret (simulate database)
users = {}

def register_user(username, password, email):
    # Hash password
    hashed_pw = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())

    # Generate Google Authenticator secret
    secret = pyotp.random_base32()

    # Save to "database"
    users[username] = {
        "password": hashed_pw,
        "secret": secret,
        "email": email
    }

    # Generate provisioning URI
    uri = pyotp.totp.TOTP(secret).provisioning_uri(
        name=email,
        issuer_name="IT312 Secure App"
    )

    # Generate QR code for user to scan
    img = qrcode.make(uri)
    print(f"✅ User '{username}' registered.")
    print("Scan this QR code in Google Authenticator app:")
    display(img)

# Example registration
#register_user("alice", "mypassword123", "alice@example.com")
register_user("demo_user", "change_me", "demo@example.com")

In [ ]:
def login(username, password):
    # Check if user exists
    if username not in users:
        print("❌ Invalid username")
        return False

    user = users[username]

    # Verify password
    if not bcrypt.checkpw(password.encode('utf-8'), user["password"]):
        print("❌ Invalid password")
        return False

    # Prompt for OTP
    totp = pyotp.TOTP(user["secret"])
    otp = input("Enter the 6-digit code from Google Authenticator: ")

    if totp.verify(otp):
        print("✅ Login successful! Welcome,", username)
        return True
    else:
        print("❌ Invalid OTP")
        return False

# Example login

duser=input("Username: ")
dpass=input("Paswword: ")
login(duser, dpass)